# Notebook 03 — t-SNE com features originais de Kim et al. (2026)

Carrega os arquivos `.npy` já extraídos do `tr_te_sets.tar` disponibilizado pelos autores no Figshare
e executa a análise t-SNE sobre o conjunto de teste balanceado (Teste B), reproduzindo a Figure 9 do artigo.

> **Nota:** o `X_test.npy` presente dentro do arquivo `tr_te_sets.tar` está corrompido
> (`ValueError: cannot reshape array of size 1834976 into shape (12, 7116673)`).
> Este notebook usa os arquivos previamente extraídos em `/content/datasets/`,
> os mesmos utilizados com sucesso no exp_2.

**Pipeline:**
1. Carrega X_test / y_test de `/content/datasets/`
2. Balanceia para Teste B (831K × 831K)
3. Amostra para t-SNE
4. t-SNE 2D
5. Métricas: Silhouette, CH, DB, Fisher Ratio
6. Plot reproduzindo a Figure 9

## 0. Configuração — ajuste o caminho do arquivo

In [ ]:
from pathlib import Path

# ── Ajuste este caminho para onde os .npy estão salvos ────────────────────────
# Google Colab (após montar o Drive ou fazer upload):
DATA_DIR = Path('/content/datasets')

# Nomes dos arquivos conforme extraídos do tr_te_sets.tar
X_TEST_FILE = DATA_DIR / 'X_test.npy'
Y_TEST_FILE = DATA_DIR / 'y_test.npy'

print(f'DATA_DIR      : {DATA_DIR}')
print(f'DATA_DIR existe: {DATA_DIR.exists()}')
print(f'X_test existe  : {X_TEST_FILE.exists()}')
print(f'y_test existe  : {Y_TEST_FILE.exists()}')

## 1. Verificar arquivos disponíveis

In [ ]:
# Lista os arquivos .npy disponíveis em DATA_DIR
if DATA_DIR.exists():
    npy_files = sorted(DATA_DIR.glob('*.npy'))
    print(f'Arquivos .npy encontrados em {DATA_DIR}:')
    for f in npy_files:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name}  ({size_mb:.1f} MB)')
else:
    raise FileNotFoundError(
        f'DATA_DIR não encontrado: {DATA_DIR}\n'
        'Certifique-se de que os arquivos .npy foram extraídos do tr_te_sets.tar\n'
        'e copiados para este diretório antes de executar o notebook.'
    )

## 2. Carregar X_test e y_test

In [ ]:
import numpy as np

# Carrega usando mmap_mode para economizar memória (shape e dtype são lidos sem
# materializar o array inteiro — útil para datasets de 3 GB+)
X_test = np.load(X_TEST_FILE, mmap_mode='r')
y_test = np.load(Y_TEST_FILE)

print(f'X_test shape : {X_test.shape}   dtype: {X_test.dtype}')
print(f'y_test shape : {y_test.shape}   dtype: {y_test.dtype}')

unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique.astype(int), counts):
    label = 'Normal' if cls == 0 else 'Ataque'
    print(f'  Classe {cls} ({label}): {cnt:,}  ({100*cnt/len(y_test):.2f}%)')

In [ ]:
# Confirma que X_test tem as 12 features comportamentais esperadas
assert X_test.ndim == 2 and X_test.shape[1] == 12, \
    f'Esperado (N, 12), obtido {X_test.shape}'

FEATURE_NAMES = [
    'IP time interval',
    'SOME/IP likelihood', 'SOME/IP-SD likelihood', 'TCP/UDP likelihood',
    'SOME/IP entropy',    'SOME/IP-SD entropy',    'TCP/UDP entropy',
    'SOME/IP payload changes', 'SOME/IP-SD payload changes', 'TCP/UDP payload changes',
    'IP length changes', 'TCP/UDP length changes',
]
print('Features (12):')
for i, name in enumerate(FEATURE_NAMES):
    print(f'  [{i:02d}] {name}')

## 3. Balancear para Teste B (se necessário)

O artigo usa o **Teste B balanceado** (831K × 831K) para o t-SNE.
Se o conjunto carregado já for balanceado, esta célula detecta e pula.

In [ ]:
from sklearn.utils import resample

def balance_test_set(X, y, seed=42):
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]
    n = min(len(idx_0), len(idx_1))

    if abs(len(idx_0) - len(idx_1)) / max(len(idx_0), len(idx_1)) < 0.05:
        print(f'[OK] Conjunto já balanceado ({len(idx_0)} vs {len(idx_1)}), sem alteração.')
        return X, y

    rng = np.random.default_rng(seed)
    s0 = rng.choice(idx_0, n, replace=False)
    s1 = rng.choice(idx_1, n, replace=False)
    idx = np.concatenate([s0, s1])
    rng.shuffle(idx)
    print(f'Balanceado: {n} Normal + {n} Ataque = {2*n:,} amostras')
    return X[idx], y[idx]

if X_test_prop is not None and y_test is not None:
    X_bal_prop, y_bal = balance_test_set(X_test_prop, y_test)
    print(f'Shape final (proposto): {X_bal_prop.shape}')

if X_test_hdr is not None:
    X_bal_hdr, _ = balance_test_set(X_test_hdr, y_test)
    print(f'Shape final (header)  : {X_bal_hdr.shape}')

In [ ]:
def balance_test_set(X, y, seed=42):
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]
    n = min(len(idx_0), len(idx_1))

    if abs(len(idx_0) - len(idx_1)) / max(len(idx_0), len(idx_1)) < 0.05:
        print(f'[OK] Conjunto já balanceado ({len(idx_0):,} vs {len(idx_1):,}), sem alteração.')
        return np.array(X), y

    rng = np.random.default_rng(seed)
    s0 = rng.choice(idx_0, n, replace=False)
    s1 = rng.choice(idx_1, n, replace=False)
    idx = np.concatenate([s0, s1])
    rng.shuffle(idx)
    print(f'Balanceado: {n:,} Normal + {n:,} Ataque = {2*n:,} amostras')
    return np.array(X[idx]), y[idx]

X_bal, y_bal = balance_test_set(X_test, y_test)
print(f'Shape final: {X_bal.shape}')

In [ ]:
N_TSNE = 20_000   # aumente se tiver memória/tempo disponível

def sample_balanced(X, y, n, seed=42):
    n_per_class = n // 2
    rng = np.random.default_rng(seed)
    idx_0 = rng.choice(np.where(y == 0)[0], min(n_per_class, (y==0).sum()), replace=False)
    idx_1 = rng.choice(np.where(y == 1)[0], min(n_per_class, (y==1).sum()), replace=False)
    idx = np.concatenate([idx_0, idx_1])
    rng.shuffle(idx)
    return X[idx], y[idx]

if X_test_prop is not None:
    X_samp_prop, y_samp = sample_balanced(X_bal_prop, y_bal, N_TSNE)
    print(f'Amostra proposto : {X_samp_prop.shape}')

if X_test_hdr is not None:
    X_samp_hdr, _  = sample_balanced(X_bal_hdr, y_bal, N_TSNE)
    print(f'Amostra header   : {X_samp_hdr.shape}')

In [ ]:
N_TSNE = 20_000   # aumente para 50K se tiver RAM/tempo disponível

def sample_balanced(X, y, n, seed=42):
    n_per_class = n // 2
    rng = np.random.default_rng(seed)
    idx_0 = rng.choice(np.where(y == 0)[0], min(n_per_class, (y==0).sum()), replace=False)
    idx_1 = rng.choice(np.where(y == 1)[0], min(n_per_class, (y==1).sum()), replace=False)
    idx = np.concatenate([idx_0, idx_1])
    rng.shuffle(idx)
    return X[idx], y[idx]

X_samp, y_samp = sample_balanced(X_bal, y_bal, N_TSNE)
print(f'Amostra para t-SNE: {X_samp.shape}  ({(y_samp==0).sum()} Normal, {(y_samp==1).sum()} Ataque)')

In [ ]:
from sklearn.manifold import TSNE
import time

TSNE_PARAMS = dict(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)

results = {}

if X_test_prop is not None:
    print('Rodando t-SNE (proposto)...')
    t0 = time.time()
    emb_prop = TSNE(**TSNE_PARAMS).fit_transform(X_samp_prop)
    print(f'  Concluído em {time.time()-t0:.1f}s')
    results['proposed'] = (emb_prop, y_samp)

if X_test_hdr is not None:
    print('Rodando t-SNE (header)...')
    t0 = time.time()
    emb_hdr = TSNE(**TSNE_PARAMS).fit_transform(X_samp_hdr)
    print(f'  Concluído em {time.time()-t0:.1f}s')
    results['header'] = (emb_hdr, y_samp)

In [ ]:
from sklearn.manifold import TSNE
import time

TSNE_PARAMS = dict(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)

print('Rodando t-SNE nas 12 features propostas...')
t0 = time.time()
emb = TSNE(**TSNE_PARAMS).fit_transform(X_samp)
elapsed = time.time() - t0
print(f'Concluído em {elapsed:.1f}s')
print(f'Embedding shape: {emb.shape}')

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def fisher_ratio_2d(emb, y):
    """Fisher Ratio nos embeddings 2D."""
    e0 = emb[y == 0]
    e1 = emb[y == 1]
    mu_diff = np.mean(e1, axis=0) - np.mean(e0, axis=0)
    between = np.dot(mu_diff, mu_diff)
    within  = np.var(e0, axis=0).sum() + np.var(e1, axis=0).sum()
    return between / within if within > 0 else 0.0

print(f'{"Métrica":<30} {"Proposto":>12} {"Header":>12}')
print('-' * 56)

# Referência do artigo (slide 12)
ref = {
    'Silhouette':         (0.0607, 0.0721),
    'Calinski-Harabasz':  (486.91, 443.48),
    'Davies-Bouldin':     (4.1696, 4.3374),
    'Fisher Ratio':       (0.1150, 0.1054),
}

metrics_our = {}
for key, (emb, y_s) in results.items():
    sil = silhouette_score(emb, y_s, sample_size=5000, random_state=42)
    ch  = calinski_harabasz_score(emb, y_s)
    db  = davies_bouldin_score(emb, y_s)
    fr  = fisher_ratio_2d(emb, y_s)
    metrics_our[key] = {'Silhouette': sil, 'Calinski-Harabasz': ch,
                        'Davies-Bouldin': db, 'Fisher Ratio': fr}

for m_name, (ref_p, ref_h) in ref.items():
    our_p = metrics_our.get('proposed', {}).get(m_name, float('nan'))
    our_h = metrics_our.get('header',   {}).get(m_name, float('nan'))
    print(f'{m_name:<30} {our_p:>12.4f} {our_h:>12.4f}')

print()
print('── Referência do artigo ──')
for m_name, (ref_p, ref_h) in ref.items():
    print(f'{m_name:<30} {ref_p:>12.4f} {ref_h:>12.4f}')

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def fisher_ratio_2d(emb, y):
    """Fisher Ratio nos embeddings 2D: between-class / within-class scatter."""
    e0 = emb[y == 0]
    e1 = emb[y == 1]
    mu_diff = np.mean(e1, axis=0) - np.mean(e0, axis=0)
    between = np.dot(mu_diff, mu_diff)
    within  = np.var(e0, axis=0).sum() + np.var(e1, axis=0).sum()
    return between / within if within > 0 else 0.0

sil = silhouette_score(emb, y_samp, sample_size=5000, random_state=42)
ch  = calinski_harabasz_score(emb, y_samp)
db  = davies_bouldin_score(emb, y_samp)
fr  = fisher_ratio_2d(emb, y_samp)

# Referência do artigo (Figure 9, panel (a) — features propostas)
ref = {
    'Silhouette':        (0.0607, sil),
    'Calinski-Harabasz': (486.91, ch),
    'Davies-Bouldin':    (4.1696, db),
    'Fisher Ratio':      (0.1150, fr),
}

print(f'{"Métrica":<26} {"Kim (2026)":>12} {"Obtido":>12} {"Δ":>10}')
print('-' * 62)
for name, (ref_val, our_val) in ref.items():
    delta = our_val - ref_val
    print(f'{name:<26} {ref_val:>12.4f} {our_val:>12.4f} {delta:>+10.4f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLOR_NORMAL = '#4472C4'   # azul
COLOR_ATTACK = '#FF0000'   # vermelho
ALPHA        = 0.35
S            = 4

n_plots = len(results)
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))
if n_plots == 1:
    axes = [axes]

titles = {'proposed': '(a) Our feature set', 'header': '(b) Packet header feature set'}

for ax, (key, (emb, y_s)) in zip(axes, results.items()):
    m = metrics_our[key]
    mask_n = y_s == 0
    mask_a = y_s == 1

    ax.scatter(emb[mask_n, 0], emb[mask_n, 1],
               c=COLOR_NORMAL, s=S, alpha=ALPHA, linewidths=0, label='Normal')
    ax.scatter(emb[mask_a, 0], emb[mask_a, 1],
               c=COLOR_ATTACK, s=S, alpha=ALPHA, linewidths=0, label='Attack')

    ax.set_title(titles.get(key, key), fontsize=12)
    ax.set_xlabel('t-SNE embedding X', fontsize=9)
    ax.set_ylabel('t-SNE embedding Y', fontsize=9)
    ax.legend(handles=[
        mpatches.Patch(color=COLOR_NORMAL, label='Normal'),
        mpatches.Patch(color=COLOR_ATTACK, label='Attack'),
    ], fontsize=8, loc='upper right')

    # Métricas no canto
    info = (f"Sil={m['Silhouette']:.4f}\n"
            f"CH={m['Calinski-Harabasz']:.1f}\n"
            f"DB={m['Davies-Bouldin']:.4f}\n"
            f"FR={m['Fisher Ratio']:.4f}")
    ax.text(0.02, 0.98, info, transform=ax.transAxes,
            fontsize=7, va='top', family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle('t-SNE — Kim et al. (2026) features originais\n'
             'Figure 9 do artigo — Teste B balanceado',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('figure9_kim_original.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figure9_kim_original.png')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLOR_NORMAL = '#4472C4'
COLOR_ATTACK = '#FF0000'
ALPHA = 0.35
S     = 4

fig, ax = plt.subplots(figsize=(7, 6))

mask_n = y_samp == 0
mask_a = y_samp == 1

ax.scatter(emb[mask_n, 0], emb[mask_n, 1],
           c=COLOR_NORMAL, s=S, alpha=ALPHA, linewidths=0, label='Normal')
ax.scatter(emb[mask_a, 0], emb[mask_a, 1],
           c=COLOR_ATTACK, s=S, alpha=ALPHA, linewidths=0, label='Attack')

ax.set_title('(a) Our feature set', fontsize=12)
ax.set_xlabel('t-SNE embedding X', fontsize=9)
ax.set_ylabel('t-SNE embedding Y', fontsize=9)
ax.legend(handles=[
    mpatches.Patch(color=COLOR_NORMAL, label='Normal'),
    mpatches.Patch(color=COLOR_ATTACK, label='Attack'),
], fontsize=8, loc='upper right')

info = (f"Sil={sil:.4f}  (Kim: 0.0607)\n"
        f"CH={ch:.1f}  (Kim: 486.91)\n"
        f"DB={db:.4f}  (Kim: 4.1696)\n"
        f"FR={fr:.4f}  (Kim: 0.1150)")
ax.text(0.02, 0.98, info, transform=ax.transAxes,
        fontsize=7, va='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle('t-SNE — Kim et al. (2026)\nReproducao Figure 9 — Teste B balanceado (features propostas)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('figure9_kim_original.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figure9_kim_original.png')

In [ ]:
rows = []
for m_name, (ref_p, ref_h) in ref.items():
    our_p = metrics_our.get('proposed', {}).get(m_name, float('nan'))
    our_h = metrics_our.get('header',   {}).get(m_name, float('nan'))
    delta_p = our_p - ref_p
    rows.append({
        'Métrica':          m_name,
        'Kim (proposto)':   ref_p,
        'Nosso (proposto)': round(our_p, 4),
        'Δ (proposto)':     round(delta_p, 4),
        'Kim (header)':     ref_h,
        'Nosso (header)':   round(our_h, 4),
    })

df_cmp = pd.DataFrame(rows).set_index('Métrica')
display(df_cmp.style.format(precision=4)
    .applymap(lambda v: 'color: green' if isinstance(v, float) and v > 0 else
                        'color: red'   if isinstance(v, float) and v < 0 else '',
              subset=['Δ (proposto)']))

In [ ]:
import pandas as pd

rows = []
for name, (ref_val, our_val) in ref.items():
    rows.append({
        'Metrica':       name,
        'Kim (2026)':    ref_val,
        'Obtido':        round(our_val, 4),
        'Delta':         round(our_val - ref_val, 4),
        'Match (<10%)?': abs(our_val - ref_val) / abs(ref_val) < 0.10,
    })

df = pd.DataFrame(rows).set_index('Metrica')
display(df)